# ERA5-Forced Sea Ice Mass Budget Sankeys

Sankey diagrams of the sea ice mass budget from an **ERA5-forced NEMO-SI3 ocean–sea ice simulation**
(eORCA025, 1/4° nominal resolution, as in Richaud et al. 2026), for comparison with the CMIP6
Sankeys in `sankey_figures_clean.ipynb`.

**Data**: `data/IceVfxClim2000-2024_1m_eORCA025_4AlekPetty.nc`, provided by Benjamin Richaud.
Monthly climatological seasonal cycle (baseline **2000–2024**) of ice mass fluxes for the two
hemispheric totals (Arctic / Antarctic). Fluxes are area-integrated rates in **kg s⁻¹**
(the `units` metadata is inherited from the raw model output; no m² involved).

To get annual budgets we integrate the monthly climatology over the year:
`Gt yr⁻¹ = Σ_month rate [kg s⁻¹] × seconds-in-month / 10¹²`.

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import os

from functions import _plotly_sankey_fig, _C

ModuleNotFoundError: No module named 'regionmask'

In [ ]:
# ── Display options ───────────────────────────────────────────────────────────
# Set SHOW_VALUES = True  to add Gt yr⁻¹ labels to each flow.
SHOW_VALUES  = False

# Set SHOW_PERCENT = True  to add each flow's % of total sources/sinks.
SHOW_PERCENT = True

# Set SHOW_RESIDUAL = False  to hide the grey Residual balancing node.
SHOW_RESIDUAL = True

# Set INCLUDE_DYNAMICS = True  to add the ice transport term (icemtrp) as Dyn. import/export.
# Near-zero for hemispheric totals (transport across the domain edge), so off by default.
INCLUDE_DYNAMICS = False# ── Display options ───────────────────────────────────────────────────────────
# Set SHOW_VALUES = True  to add Gt yr⁻¹ labels to each flow.
SHOW_VALUES  = False

# Set SHOW_PERCENT = True  to add each flow's % of total sources/sinks.
SHOW_PERCENT = True

# Set SHOW_RESIDUAL = False  to hide the grey Residual balancing node.
SHOW_RESIDUAL = True

# Set INCLUDE_DYNAMICS = True  to add the ice transport term (icemtrp) as Dyn. import/export.
# Near-zero for hemispheric totals (transport across the domain edge), so off by default.
INCLUDE_DYNAMICS = False

## Load Data

In [ ]:
ds = xr.open_dataset('files/IceVfxClim2000-2024_1m_eORCA025_4AlekPetty.nc')
ds

## Convert to Annual Budgets (Gt yr⁻¹)

Each variable is a climatological mean rate (kg s⁻¹) per month, so the annual total is the
day-weighted sum over the seasonal cycle. February uses 28.25 days (leap years within 2000–2024).

In [ ]:
days_in_month = xr.DataArray([31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31],
                             dims='month', coords={'month': ds.month})

annual = {v: (ds[v] * days_in_month * 86400 / 1e12).sum('month') for v in ds.data_vars}

budget_table = pd.DataFrame({v: annual[v].values for v in annual}, index=ds.hemisphere.values).T
budget_table.columns = [f"{h} (Gt yr⁻¹)" for h in ds.hemisphere.values]
budget_table.round(1)days_in_month = xr.DataArray([31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31],
                             dims='month', coords={'month': ds.month})

annual = {v: (ds[v] * days_in_month * 86400 / 1e12).sum('month') for v in ds.data_vars}

budget_table = pd.DataFrame({v: annual[v].values for v in annual}, index=ds.hemisphere.values).T
budget_table.columns = [f"{h} (Gt yr⁻¹)" for h in ds.hemisphere.values]
budget_table.round(1)

In [ ]:
# Closure check: 'total' should equal the sum of all component fluxes (incl. transport).
components = sum(annual[v] for v in ds.data_vars if v != 'total')
assert np.allclose(components, annual['total'], atol=1.0), "Budget does not close!"
print("Budget closes: total = sum of components")
print("Net annual ice mass change:", annual['total'].values.round(1), "Gt yr⁻¹ (Arctic, Antarctic)")

## Term Mapping

| NEMO-SI3 variable | Sankey flow | Notes |
|---|---|---|
| `vfxbog` | Basal growth | |
| `vfxopw` | Open water ice production | frazil |
| `vfxsni` | Snow→ice | snow-ice formation |
| `vfxdyn` | Ridging growth | ocean water freezing during ridge consolidation — sizable in NEMO (~1400 Gt yr⁻¹ Arctic), unlike CMIP6 models where ridging conserves ice mass |
| `icemtrp` | Dyn. import / export | transport across the hemispheric domain edge (near zero for hemispheric totals) |
| `vfxbom` | Basal melt | |
| `vfxsum` + `vfxpnd` | Top melt | melt ponds folded into surface melt (net ≈ 0 annually) |
| `vfxlam` | Lateral melt | |
| `vfxsub` | Evap/subl | sublimation |
| `vfxres` | — | undiagnosed processes; absorbed by the Residual node together with the net annual mass change |

Sign convention in the file: growth positive, melt negative. The automatic Residual node therefore
represents net annual ice mass change plus `vfxres`.

In [ ]:

os.makedirs("figures", exist_ok=True)

HEMIS = {
    'Arctic':    dict(abbrev='AO', subtitle='Arctic Ocean'),
    'Antarctic': dict(abbrev='SO', subtitle='Southern Ocean'),
}

for hemi, info in HEMIS.items():
    a = {v: float(annual[v].sel(hemisphere=hemi)) for v in annual}

    inflows = [
        (max(0.0,  a['vfxsni']),               "Snow→ice",                  _C["snow2ice_g"]),
        (max(0.0,  a['vfxopw']),               "Open water ice production", _C["frazil"]),
        (max(0.0,  a['vfxbog']),               "Basal growth",              _C["basal_g"]),
        (max(0.0,  a['vfxdyn']),               "Ridging growth",            _C["dyn_in"]),
    ]
    if INCLUDE_DYNAMICS:
        inflows.append((max(0.0, a['icemtrp']), "Dyn. import", _C["dyn_in"]))
    outflows = [
        (max(0.0, -a['vfxsub']),               "Evap/subl",    _C["evapsubl"]),
        (max(0.0, -(a['vfxsum'] + a['vfxpnd'])), "Top melt",   _C["top_melt"]),
        (max(0.0, -a['vfxdyn']),               "Ridging loss", _C["dyn_out"]),
        (max(0.0, -a['vfxlam']),               "Lateral melt", _C["lat_melt"]),
        (max(0.0, -a['vfxbom']),               "Basal melt",   _C["basal_melt"]),
    ]

    if INCLUDE_DYNAMICS:
        outflows.insert(2, (max(0.0, -a['icemtrp']), "Dyn. export", _C["dyn_out"]))

    inflow_group  = {"members": {"Open water ice production", "Basal growth"}, "color": _C["ice_growth"], "name": None}
    outflow_group = {"members": {"Basal melt", "Lateral melt"}, "color": _C["ice_melt"], "name": None}

    fig = _plotly_sankey_fig(
        "Sea Ice", "#FFF59D", inflows, outflows,
        "ERA5-forced NEMO-SI3", "Sea Ice Mass",
        inflow_group=inflow_group, outflow_group=outflow_group,
        show_values=SHOW_VALUES, show_residual=SHOW_RESIDUAL, show_percent=SHOW_PERCENT,
        hide_terminal_nodes=True,
        subtitle=info['subtitle'],
    )
    fig.write_image(f"figures/ice_sankey_{info['abbrev']}_ERA5_2000-2024.png", scale=3.125)
    fig.write_image(f"figures/ice_sankey_{info['abbrev']}_ERA5_2000-2024.pdf")
    fig.show()import os
os.makedirs("figures", exist_ok=True)

HEMIS = {
    'Arctic':    dict(abbrev='AO', subtitle='Arctic Ocean'),
    'Antarctic': dict(abbrev='SO', subtitle='Southern Ocean'),
}

for hemi, info in HEMIS.items():
    a = {v: float(annual[v].sel(hemisphere=hemi)) for v in annual}

    inflows = [
        (max(0.0,  a['vfxsni']),               "Snow→ice",                  _C["snow2ice_g"]),
        (max(0.0,  a['vfxopw']),               "Open water ice production", _C["frazil"]),
        (max(0.0,  a['vfxbog']),               "Basal growth",              _C["basal_g"]),
        (max(0.0,  a['vfxdyn']),               "Ridging growth",            _C["dyn_in"]),
    ]
    if INCLUDE_DYNAMICS:
        inflows.append((max(0.0, a['icemtrp']), "Dyn. import", _C["dyn_in"]))
    outflows = [
        (max(0.0, -a['vfxsub']),               "Evap/subl",    _C["evapsubl"]),
        (max(0.0, -(a['vfxsum'] + a['vfxpnd'])), "Top melt",   _C["top_melt"]),
        (max(0.0, -a['vfxdyn']),               "Ridging loss", _C["dyn_out"]),
        (max(0.0, -a['vfxlam']),               "Lateral melt", _C["lat_melt"]),
        (max(0.0, -a['vfxbom']),               "Basal melt",   _C["basal_melt"]),
    ]

    if INCLUDE_DYNAMICS:
        outflows.insert(2, (max(0.0, -a['icemtrp']), "Dyn. export", _C["dyn_out"]))

    inflow_group  = {"members": {"Open water ice production", "Basal growth"}, "color": _C["ice_growth"], "name": None}
    outflow_group = {"members": {"Basal melt", "Lateral melt"}, "color": _C["ice_melt"], "name": None}

    fig = _plotly_sankey_fig(
        "Sea Ice", "#FFF59D", inflows, outflows,
        "ERA5-forced NEMO-SI3", "Sea Ice Mass",
        inflow_group=inflow_group, outflow_group=outflow_group,
        show_values=SHOW_VALUES, show_residual=SHOW_RESIDUAL, show_percent=SHOW_PERCENT,
        hide_terminal_nodes=True,
        subtitle=info['subtitle'],
    )
    fig.write_image(f"figures/ice_sankey_{info['abbrev']}_ERA5_2000-2024.png", scale=3.125)
    fig.write_image(f"figures/ice_sankey_{info['abbrev']}_ERA5_2000-2024.pdf")
    fig.show()